# Component B, Question 3: Machine Learning for Navigation and Decision-Making

**Goal.** Replace the hand-written navigation rule with a **neural network**
that decides each step whether to go FORWARD, turn LEFT or turn RIGHT, using
only what the robot senses: 8 LiDAR sector distances plus the direction to
its current goal.

**Method: imitation learning (behavioural cloning).** Every step of every
robot in the twelve rule-driven runs was logged with the rule's decision,
giving a large set of labelled examples (counted below). A multi-layer perceptron is trained
to reproduce the decisions, evaluated offline, and then **deployed in Webots**
to drive the swarm itself, which is the real test.

The second half compares **centralised and distributed decision-making**
using the four coordination experiments, including the two failure tests.

In [ ]:
import sys
sys.path.append("..")
from pathlib import Path
import math

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

import swarm_data as sd

OUT_DIR = Path("outputs")
OUT_DIR.mkdir(exist_ok=True)
plt.rcParams.update({"figure.dpi": 110})

In [ ]:
from sklearn.metrics import (ConfusionMatrixDisplay, accuracy_score,
                             confusion_matrix, f1_score)
from sklearn.neural_network import MLPClassifier
from sklearn.preprocessing import StandardScaler
import json

FEATURES = sd.SENSOR_COLS + ["sin_bearing", "cos_bearing"]
CLASSES = ["FORWARD", "LEFT", "RIGHT"]

frames = []
for e in sd.BASE_EXPERIMENTS:
    for i in range(sd.N_ROBOTS):
        d = sd.robot_log(e, i)
        d = d[d.expert_action.isin(CLASSES)].copy()
        d["experiment"] = e
        d["seed"] = int(e.rsplit("_s", 1)[1])
        frames.append(d)
data = pd.concat(frames, ignore_index=True)
data["sin_bearing"] = np.sin(data.bearing)
data["cos_bearing"] = np.cos(data.bearing)
X, y = data[FEATURES].values, data.expert_action.values

print(f"{len(data)} labelled decisions from {len(sd.BASE_EXPERIMENTS)} experiments")
data.groupby("seed").size().rename("samples").to_frame().T

In [ ]:
data.expert_action.value_counts(normalize=True).round(3).rename("class share").to_frame().T

## Model and evaluation protocol

- **Inputs (10):** 8 LiDAR sector minima (0–1 m), plus sin and cos of the goal bearing. Sin/cos avoids the jump at ±180° that a raw angle would have. Inputs are standardised.
- **Network:** a multi-layer perceptron with hidden layers of 64 and 32 ReLU units and a softmax output over the 3 actions. It is trained with Adam, and 10% of the training data is held out for early stopping.
- **Leave-one-seed-out validation.** Consecutive steps in one run are nearly identical, so a random train/test split would leak information and overstate accuracy. Instead, all runs of one seed (4 runs) are held out in turn, and the network is trained on the other 8. This tests generalisation to *runs it has never seen*.

In [ ]:
def make_model():
    return MLPClassifier(hidden_layer_sizes=(64, 32), max_iter=400,
                         early_stopping=True, n_iter_no_change=20,
                         random_state=0)


fold_rows, y_true_all, y_pred_all = [], [], []
for held_out in sd.SEEDS:
    test = (data.seed == held_out).values
    scaler = StandardScaler().fit(X[~test])
    model = make_model().fit(scaler.transform(X[~test]), y[~test])
    pred = model.predict(scaler.transform(X[test]))
    y_true_all += list(y[test])
    y_pred_all += list(pred)
    fold_rows.append({"Held-out seed": held_out,
                      "Train accuracy": round(accuracy_score(
                          y[~test], model.predict(scaler.transform(X[~test]))), 3),
                      "Test accuracy": round(accuracy_score(y[test], pred), 3),
                      "Test macro-F1": round(f1_score(y[test], pred,
                                                      average="macro"), 3)})
folds = pd.DataFrame(fold_rows)
folds.loc["mean"] = ["mean", folds["Train accuracy"].mean().round(3),
                     folds["Test accuracy"].mean().round(3),
                     folds["Test macro-F1"].mean().round(3)]
folds

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4.5))
ConfusionMatrixDisplay(confusion_matrix(y_true_all, y_pred_all, labels=CLASSES,
                                        normalize="true"),
                       display_labels=CLASSES).plot(ax=axes[0], cmap="Blues",
                                                    values_format=".2f",
                                                    colorbar=False)
axes[0].set_title("Confusion matrix (all held-out folds, row-normalised)")
axes[0].set_xlabel("network's action")
axes[0].set_ylabel("rule's action")

scaler = StandardScaler().fit(X)
final_model = make_model().fit(scaler.transform(X), y)
axes[1].plot(final_model.loss_curve_, label="training loss")
ax2 = axes[1].twinx()
ax2.plot(final_model.validation_scores_, c="tab:orange",
         label="validation accuracy")
axes[1].set_xlabel("epoch")
axes[1].set_ylabel("cross-entropy loss")
ax2.set_ylabel("validation accuracy")
axes[1].set_title("Training curve (final model, all data)")
axes[1].legend(loc="center right", fontsize=8)
ax2.legend(loc="lower right", fontsize=8)
plt.tight_layout()
plt.savefig(OUT_DIR / "b3_confusion_training.png", bbox_inches="tight")
plt.show()

In [ ]:
# Where does the network disagree with the rule?
blocked = (data.s0 < 0.12) | (np.minimum(data.s1, data.s7) < 0.08)
pred_all = pd.Series(y_pred_all, index=pd.concat(
    [data[data.seed == s] for s in sd.SEEDS]).index)
data["held_out_pred"] = pred_all
data["correct"] = data.held_out_pred == data.expert_action
pd.DataFrame({
    "Situation": ["Way ahead clear", "Blocked (avoiding)"],
    "Share of data": [round((~blocked).mean(), 3), round(blocked.mean(), 3)],
    "Held-out accuracy": [round(data.correct[~blocked].mean(), 3),
                          round(data.correct[blocked].mean(), 3)],
})

**Interpretation (offline).** The network learns the rule well, but not
perfectly. FORWARD is almost always right. Most errors are **LEFT ↔ RIGHT
confusions**, and they are more frequent while avoiding obstacles. The rule
contains hard thresholds ("blocked if closer than 12 cm", "turn right only if
the right side is clearly more open"), and a smooth network can only
approximate those sharp boundaries. Training accuracy is higher than held-out
accuracy, so part of the gap is also situations the network has not seen.

### Export for deployment
The trained weights are written to `webots/data/swarm/nn_policy.json`. The
robot controller runs the forward pass in plain Python, so no machine-learning
library is needed inside Webots. The next press of Play runs the
neural-network experiment automatically.

In [ ]:
policy = {"mean": scaler.mean_.tolist(), "scale": scaler.scale_.tolist(),
          "weights": [w.tolist() for w in final_model.coefs_],
          "biases": [b.tolist() for b in final_model.intercepts_],
          "classes": final_model.classes_.tolist(), "features": FEATURES}
with open(sd.DATA_DIR / "nn_policy.json", "w") as f:
    json.dump(policy, f)


# Check: the plain-Python forward pass used on the robot matches sklearn.
def forward(features):
    h = (np.asarray(features) - scaler.mean_) / scaler.scale_
    for k, (W, b) in enumerate(zip(final_model.coefs_,
                                   final_model.intercepts_)):
        h = h @ W + b
        if k < len(final_model.coefs_) - 1:
            h = np.maximum(h, 0)
    return final_model.classes_[np.argmax(h)]


sample = np.random.default_rng(0).choice(len(X), 1000, replace=False)
agree = np.mean([forward(X[i]) == final_model.predict(
    scaler.transform(X[i:i + 1]))[0] for i in sample])
print(f"Saved {sd.DATA_DIR / 'nn_policy.json'}; robot forward pass agrees "
      f"with sklearn on {agree:.1%} of 1000 samples")

## In-simulation test: the network drives the swarm

The `distributed_nn` runs (3 seeds) repeat the distributed experiment with
the same starts, seeds and coordination, but with the network choosing every
action. The commitment layer (spin while blocked, then escape forward) is
shared by both policies, so the comparison is fair.

In [ ]:
if not sd.runs("distributed_nn"):
    print("Run the Webots world again to produce the neural-network runs.")
else:
    def closed_loop_agreement(run):
        rates = []
        for i in range(sd.N_ROBOTS):
            d = sd.robot_log(run, i)
            d = d[d.action.isin(CLASSES)]
            rates.append((d.action == d.expert_action).mean())
        return 100 * np.mean(rates)

    in_sim = sd.kind_table(["distributed", "distributed_nn"],
                           ["gbest error (cm)", "contact events",
                            "closest gap (mm)"])
    agreement = {k: [closed_loop_agreement(r) for r in sd.runs(k)]
                 for k in ["distributed", "distributed_nn"]}
    in_sim.insert(0, "agreement with rule while driving (%)",
                  [f"{np.mean(v):.1f} ± {np.std(v, ddof=1):.1f}"
                   for v in agreement.values()])
    in_sim.index = ["Rule (teacher)", "Neural network"]
    in_sim.to_csv(OUT_DIR / "b3_in_simulation.csv")
    display(in_sim)

    fig, axes = plt.subplots(1, 2, figsize=(12, 5.5))
    for ax, (e, label) in zip(axes, [("distributed_s1", "Rule"),
                                     ("distributed_nn_s1", "Neural network")]):
        sd.draw_arena(ax)
        for i in range(sd.N_ROBOTS):
            p = sd.robot_log(e, i).query("phase != 'consensus'")
            ax.plot(p.x, p.y, color=sd.ROBOT_COLOURS[i], lw=1.2)
        ax.set_title(f"{label} driving, seed 1 (formation + search)")
    plt.tight_layout()
    plt.savefig(OUT_DIR / "b3_nn_vs_rule.png", bbox_inches="tight")
    plt.show()

**Interpretation (in simulation).** A well-known risk with behavioural
cloning is **compounding error**. Offline, every input came from the
teacher's own trajectories. Once the network drives, its small mistakes can
move the robot into situations the teacher never produced, where further
mistakes become more likely. Comparing the network's agreement *while
driving* with its offline held-out accuracy tests for this. Here the two are
close, so there is no sign of serious compounding error. The swarm found the
source about as accurately as with the rule, with no contacts. The network
also kept slightly larger gaps from other robots, which suggests it learned a
smoothed, slightly more cautious version of the rule's sharp thresholds.

Two factors plausibly explain this. The training data covered a wide range
of situations (twelve runs, including failures). And the shared commitment
and safety layer stops a wrong turn from cascading into a trap.

The important limitation is that **the network was only tested in the arena
it was trained in**. Different obstacle shapes, more robots, or narrower gaps
would produce inputs outside the training data, and nothing here shows how
the network would behave there. A simple, verifiable safety layer around the
learned component is the practical safeguard. Where possible, testing should
use environments the model has never seen (iterative retraining such as DAgger
can help close any gap found).

## Centralised vs distributed decision-making

In [ ]:
fault_start = 5     # PSO iteration at which the fault is injected
fig, axes = plt.subplots(1, 2, figsize=(13, 4.5))
for ax, pair in zip(axes, [("distributed_s1", "distributed_robot_failure_s1"),
                           ("centralised_s1",
                            "centralised_coordinator_failure_s1")]):
    for e, ls in zip(pair, ["-", "--"]):
        prog = sd.pso_progress(e)
        for i in range(sd.N_ROBOTS):
            r = prog[prog.robot == i]
            ax.plot(r.iteration, 100 * r.gbest_error, ls,
                    color=sd.ROBOT_COLOURS[i],
                    label=f"robot_{i}" if ls == "-" else None)
    ax.axvline(fault_start, c="k", lw=1)
    ax.text(fault_start + 0.2, ax.get_ylim()[1] * 0.92, "fault", fontsize=8)
    ax.set_xlabel("PSO iteration")
    ax.set_ylabel("gbest distance to source (cm)")
    ax.legend(fontsize=7)
axes[0].set_title("Distributed, seed 1: normal (solid) vs robot_2 fails (dashed)")
axes[1].set_title("Centralised, seed 1: normal (solid) vs coordinator fails (dashed)")
plt.tight_layout()
plt.savefig(OUT_DIR / "b3_failure_comparison.png", bbox_inches="tight")
plt.show()

In [ ]:
coordination = sd.kind_table(sd.KINDS, ["gbest error (cm)",
                                        "all within 10 cm (iter)",
                                        "messages", "contact events"])
coordination.insert(0, "single point of failure",
                    ["no", "yes", "no", "yes"])
coordination.to_csv(OUT_DIR / "b3_coordination.csv")
coordination

**Interpretation.**

| | Centralised (coordinator) | Distributed (ring) |
|---|---|---|
| Information speed | Everyone learns the global best in the same iteration | Spreads one neighbour per iteration; the last robot learns about 2 iterations later |
| Communication | Fewer messages, but all through one node | More messages, all local |
| Normal accuracy | About 6 cm | About 5 cm (no meaningful difference given the seed-to-seed spread) |
| Robot failure | Not tested; the coordinator would simply stop hearing from that robot | The ring becomes a line; the survivors are just as accurate |
| Coordinator failure | **Shared knowledge is lost.** Each robot can only use its own discoveries: the error is several times larger and varies widely between seeds, and in no run did all robots reach 10 cm | No coordinator to lose |
| Scalability | Coordinator load grows with swarm size | Each robot's load is fixed by its number of neighbours |

The coordinator spreads information faster and with fewer messages, but it is
a **single point of failure**: when it stopped responding, the swarm lost its
shared knowledge and never fully converged. The distributed swarm spreads
information more slowly, but it degrades *gracefully*. Losing a robot removed
one searcher without harming the others' accuracy. For safety-critical
deployments such as search and rescue, graceful degradation is usually worth
the extra messages. A practical design often combines the two: distributed by
default, with a coordinator used opportunistically while it is reachable.